# FINALSCRAPING2 - CNBC 1 Sep 2021 s/d 1 Sep 2026

**Cara pakai (4 orang, masing-masing ~33.016 URL):**
1. Satu orang jalankan cell *Split URL* sekali, lalu commit/share `urls_shard_1..4.csv`.
2. Tiap orang: set `SHARD_ID` (1-4) di cell config, Run All sampai cell *Runner*. Boleh dihentikan & dilanjutkan kapan saja (auto-resume).
3. Setelah semua shard selesai: satu orang jalankan cell *Merge & QA*.

Jangan commit: `*.db`, `failures_*.csv`, `blocks_*.csv`, `summary_*.json`, `cnbc_articles.csv`.


In [ ]:
import csv, glob, json, random, re, sqlite3, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

SHARD_ID = 1          # <- ubah ke 1, 2, 3, atau 4
N_WORKERS = 8
RATE_START = 2.0      # req/s per orang (agregat tim = 4x; naikkan bila beda jaringan)
RATE_CAP = 5.0
MAX_ATTEMPTS = 3
COOLDOWN_SEC = 300

SITEMAP_CSV = "merged_sitemaps.csv"
SORTED_CSV = "urls_sorted.csv"
SHARD_CSV = f"urls_shard_{SHARD_ID}.csv"
DB_PATH = f"cnbc_articles_shard_{SHARD_ID}.db"
FAIL_CSV = f"failures_shard_{SHARD_ID}.csv"
BLOCK_CSV = f"blocks_shard_{SHARD_ID}.csv"
SUMMARY_JSON = f"summary_shard_{SHARD_ID}.json"

START = pd.Timestamp("2021-09-01", tz="UTC")
END = pd.Timestamp("2026-09-01", tz="UTC")

HEADERS = {
    "User-Agent": {
        1: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        2: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36",
        3: "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
        4: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
    }[SHARD_ID],
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
}


## 1. Split URL (otomatis, butuh `merged_sitemaps.csv`)

Filter tanggal terbit dari URL + bagi 4 bagian kontigu setelah diurutkan.


In [ ]:
def build_shards():
    df = pd.read_csv(SITEMAP_CSV).drop_duplicates("loc")
    df = df[df["loc"].str.endswith(".html")]
    pub = pd.to_datetime(df["loc"].str.extract(r"cnbc\.com/(\d{4}/\d{2}/\d{2})/")[0], utc=True, errors="coerce")
    df = df.assign(publish_date=pub).dropna(subset=["publish_date"])
    df = df[(df["publish_date"] >= START) & (df["publish_date"] <= END)].sort_values("publish_date").reset_index(drop=True)
    size = -(-len(df) // 4)
    df["shard"] = df.index // size + 1
    df[["loc", "publish_date", "shard"]].to_csv(SORTED_CSV, index=False)
    for s in range(1, 5):
        df.loc[df["shard"] == s, ["loc", "publish_date"]].to_csv(f"urls_shard_{s}.csv", index=False)
    print(f"total {len(df)} URL | per shard: {df['shard'].value_counts().sort_index().to_dict()}")


if not Path(SHARD_CSV).exists():
    build_shards()
else:
    print(f"{SHARD_CSV} sudah ada, lewati split. Total: {len(pd.read_csv(SHARD_CSV))}")


## 2. Parser (author string/list diperbaiki, status paywall ditandai)


In [ ]:
def parse_author(author):
    if isinstance(author, str):
        return author
    if isinstance(author, dict):
        return author.get("name", "")
    if isinstance(author, list):
        return "; ".join(a if isinstance(a, str) else a.get("name", "") for a in author if isinstance(a, (str, dict)))
    return ""


def extract_categories(html):
    cats = {}
    for raw in re.findall(r'\{"headline":"[^{}]*"__typename":"tag"\}', html):
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if obj.get("type") == "franchise":
            key = obj.get("id") or obj.get("headline")
            cats[key] = obj.get("tagName") or obj.get("headline")
    return list(cats.values())


def parse_article(url, html):
    soup = BeautifulSoup(html, "html.parser")
    ld = {}
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            obj = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        if isinstance(obj, dict) and obj.get("@type") == "NewsArticle":
            ld = obj
            break

    bodies = soup.select("div[class*='ArticleBody']")
    body = max(bodies, key=lambda d: len(d.find_all("p")), default=None)
    paragraphs = [p.get_text(" ", strip=True) for p in (body.find_all("p") if body else [])]
    content = "\n".join(t for t in paragraphs if t and t != "In this article")

    return {
        "url": url,
        "title": ld.get("headline") or "",
        "author": parse_author(ld.get("author")),
        "date_published": ld.get("datePublished") or "",
        "date_modified": ld.get("dateModified") or "",
        "categories": extract_categories(html),
        "content": content,
        "content_status": "ok" if content else ("paywalled" if body else "missing"),
    }


BLOCK_MARKERS = ("access denied", "unusual traffic", "captcha", "are you a robot", "request blocked")


def looks_blocked(text):
    if len(text) > 50000:
        return False
    low = text.lower()
    return any(m in low for m in BLOCK_MARKERS)


## 3. Database + resume (jalankan ulang aman)


In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS articles (
    url TEXT PRIMARY KEY, title TEXT, author TEXT,
    date_published TEXT, date_modified TEXT, content TEXT,
    content_status TEXT, http_status INTEGER, fetched_at TEXT
);
CREATE TABLE IF NOT EXISTS article_categories (
    url TEXT, category TEXT, PRIMARY KEY (url, category)
);
CREATE TABLE IF NOT EXISTS scrape_failures (
    url TEXT PRIMARY KEY, attempts INTEGER, last_status INTEGER,
    error TEXT, last_attempt_at TEXT
);
"""


def open_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.executescript(SCHEMA)
    conn.commit()
    return conn


conn = open_db()
done = {r[0] for r in conn.execute("SELECT url FROM articles")}
failed = {u: (a, s) for u, a, s in conn.execute("SELECT url, attempts, last_status FROM scrape_failures")}
urls = pd.read_csv(SHARD_CSV)["loc"].tolist()
pending = [
    u for u in urls
    if u not in done and failed.get(u, (0, 0))[0] < MAX_ATTEMPTS and failed.get(u, (0, 0))[1] not in (404, 410)
]
random.seed(SHARD_ID)
random.shuffle(pending)
print(f"shard {SHARD_ID}: {len(urls)} URL | selesai {len(done)} | akan diskrape {len(pending)}")


## 4. Runner (concurrent + rate adaptif + circuit breaker)


In [ ]:
class RateLimiter:
    def __init__(self, rate):
        self.rate = rate
        self.lock = threading.Lock()
        self.next_time = time.monotonic()

    def wait(self):
        with self.lock:
            now = time.monotonic()
            self.next_time = max(self.next_time, now)
            delay = self.next_time - now
            self.next_time += 1.0 / self.rate
        if delay > 0:
            time.sleep(delay)

    def slow_down(self):
        with self.lock:
            self.rate = max(0.5, self.rate / 2)

    def speed_up(self):
        with self.lock:
            self.rate = min(RATE_CAP, self.rate + 0.25)

    def force(self, rate):
        with self.lock:
            self.rate = rate


limiter = RateLimiter(RATE_START)
local = threading.local()

try:
    COOKIES = requests.get("https://www.cnbc.com/", headers=HEADERS, timeout=15).cookies.get_dict()
    print("warm-up ok, cookies:", len(COOKIES))
except Exception as e:
    COOKIES = {}
    print("warm-up gagal (lanjut tanpa cookie):", e)


def get_session():
    if not hasattr(local, "session"):
        s = requests.Session()
        s.headers.update(HEADERS)
        if COOKIES:
            s.cookies.update(COOKIES)
        local.session = s
    return local.session


def fetch(url):
    status, error = 0, ""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            limiter.wait()
            r = get_session().get(url, timeout=20)
            status = r.status_code
            if status == 200:
                if not looks_blocked(r.text):
                    return r.text, 200, ""
                status, error = 0, "soft-block"
            elif status in (404, 410):
                return None, status, "dead link"
            else:
                error = f"http {status}"
                retry_after = r.headers.get("Retry-After", "")
                if retry_after.isdigit():
                    time.sleep(min(int(retry_after), 120))
        except Exception as e:
            status, error = 0, type(e).__name__
        limiter.slow_down()
        time.sleep(2 ** (attempt - 1) + random.random())
    return None, status, error


def save_success(item, status):
    now = datetime.now(timezone.utc).isoformat()
    conn.execute(
        "INSERT OR REPLACE INTO articles (url, title, author, date_published, date_modified, "
        "content, content_status, http_status, fetched_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (item["url"], item["title"], item["author"], item["date_published"], item["date_modified"],
         item["content"], item["content_status"], status, now),
    )
    conn.execute("DELETE FROM article_categories WHERE url = ?", (item["url"],))
    for cat in item["categories"]:
        conn.execute("INSERT OR IGNORE INTO article_categories (url, category) VALUES (?, ?)", (item["url"], cat))
    conn.execute("DELETE FROM scrape_failures WHERE url = ?", (item["url"],))


def save_failure(url, status, error):
    now = datetime.now(timezone.utc).isoformat()
    attempts = failed.get(url, (0, 0))[0] + 1
    failed[url] = (attempts, status)
    conn.execute(
        "INSERT INTO scrape_failures (url, attempts, last_status, error, last_attempt_at) VALUES (?, ?, ?, ?, ?) "
        "ON CONFLICT(url) DO UPDATE SET attempts=excluded.attempts, last_status=excluded.last_status, "
        "error=excluded.error, last_attempt_at=excluded.last_attempt_at",
        (url, attempts, status, error, now),
    )
    with open(FAIL_CSV, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([url, attempts, status, error, now])


if not Path(FAIL_CSV).exists():
    with open(FAIL_CSV, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["url", "attempts", "status", "error", "time"])
if not Path(BLOCK_CSV).exists():
    with open(BLOCK_CSV, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["time", "status", "error", "rate"])

success, dead, fail, blocks, consecutive = 0, 0, 0, 0, 0
content_counts = Counter()
started = time.time()
total = len(pending)

with ThreadPoolExecutor(N_WORKERS) as pool:
    futures = {pool.submit(fetch, u): u for u in pending}
    for i, fut in enumerate(as_completed(futures), 1):
        url = futures[fut]
        try:
            html, status, error = fut.result()
        except Exception as e:
            html, status, error = None, 0, type(e).__name__
        if html is not None:
            article = parse_article(url, html)
            save_success(article, status)
            success += 1
            content_counts[article["content_status"]] += 1
            consecutive = 0
            if success % 500 == 0:
                limiter.speed_up()
        elif status in (404, 410):
            save_failure(url, status, error)
            dead += 1
            consecutive = 0
        else:
            save_failure(url, status, error)
            fail += 1
            if status in (403, 429, 503) or error == "soft-block":
                consecutive += 1
                with open(BLOCK_CSV, "a", newline="", encoding="utf-8") as f:
                    csv.writer(f).writerow([datetime.now(timezone.utc).isoformat(), status, error, round(limiter.rate, 2)])
            else:
                consecutive = 0
        if i % 25 == 0:
            conn.commit()
        if consecutive >= 5:
            blocks += 1
            limiter.force(0.5)
            print(f"!! indikasi block x{consecutive} -> cooldown {COOLDOWN_SEC}s, rate 0.5/s")
            time.sleep(COOLDOWN_SEC)
            consecutive = 0
        if i % 100 == 0:
            elapsed = time.time() - started
            print(f"[{i}/{total}] ok={success} dead={dead} fail={fail} | {i / elapsed:.1f} url/s | ETA {(total - i) / (i / elapsed) / 60:.0f} mnt")
        if blocks >= 3:
            print("!! 3x cooldown - stop. Jalankan ulang notebook nanti (auto-resume).")
            pool.shutdown(wait=False, cancel_futures=True)
            break

conn.commit()
summary = {
    "shard": SHARD_ID, "urls": len(urls), "already_done": len(done), "attempted": total,
    "success": success, "dead": dead, "failed": fail,
    "content_status": dict(content_counts), "block_cooldowns": blocks,
    "seconds": round(time.time() - started, 1),
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(summary)


## 5. Merge & QA (jalankan setelah 4 shard selesai; pastikan cell 1 dan 3 sudah dijalankan)


In [ ]:
FINAL_DB = "cnbc_articles.db"
main = open_db(FINAL_DB)

for path in sorted(glob.glob("cnbc_articles_shard_*.db")):
    main.execute("ATTACH DATABASE ? AS shard", (path,))
    main.execute("INSERT OR IGNORE INTO articles SELECT * FROM shard.articles")
    main.execute("INSERT OR IGNORE INTO article_categories SELECT * FROM shard.article_categories")
    main.execute("INSERT OR IGNORE INTO scrape_failures SELECT * FROM shard.scrape_failures")
    main.commit()
    main.execute("DETACH DATABASE shard")

df = pd.read_sql(
    "SELECT a.url, a.title, a.author, a.date_published, a.date_modified, "
    "group_concat(c.category, '; ') AS categories, a.content, a.content_status "
    "FROM articles a LEFT JOIN article_categories c ON a.url = c.url "
    "GROUP BY a.url ORDER BY a.date_published",
    main,
)
df.to_csv("cnbc_articles.csv", index=False, quoting=csv.QUOTE_ALL)

fail_files = sorted(glob.glob("failures_shard_*.csv"))
if fail_files:
    fails = pd.concat([pd.read_csv(p) for p in fail_files], ignore_index=True).drop_duplicates("url")
    fails.to_csv("failures_all.csv", index=False)
    fails[fails["status"].isin([404, 410])].to_csv("dead_links.csv", index=False)

report = []
for path in sorted(glob.glob("cnbc_articles_shard_*.db")):
    c = sqlite3.connect(path)
    report.append({
        "shard": path.split("shard_")[-1].split(".")[0],
        "articles": c.execute("SELECT count(*) FROM articles").fetchone()[0],
        "paywalled": c.execute("SELECT count(*) FROM articles WHERE content_status='paywalled'").fetchone()[0],
        "failures": c.execute("SELECT count(*) FROM scrape_failures").fetchone()[0],
    })
    c.close()
report = pd.DataFrame(report)
report.to_csv("scrape_report.csv", index=False)

print(f"total artikel: {len(df)} | duplikat: {df['url'].duplicated().sum()}")
print(df["content_status"].value_counts().to_string())
print(report.to_string(index=False))
